# 00 · Smoke Test de Reconocimiento del Habla (ASR) con Whisper

**Proyecto:** Traducción de Mensajes de Audio con Clonación de Voz mediante IA  
**Módulo:** Reconocimiento Automático del Habla (ASR)  
**Modelo:** `openai/whisper-small`  
**Audio:** Español latinoamericano (`es_419`) mono 16 kHz  

> **AVISO IMPORTANTE:** Este notebook realiza un **SMOKE TEST — 3 MUESTRAS. NO ES UNA EVALUACIÓN GLOBAL DE WHISPER.** Su propósito es comprobar que la carga del modelo, el preprocesamiento de audio a 16 kHz mono y la inferencia sin traducción operan correctamente.

## 1. Configuración del Entorno y GPU

In [1]:
# 1. Instalar dependencias si se ejecuta en Google Colab
%pip install --quiet "transformers>=4.38.0" "datasets>=3.0.0" "soundfile>=0.12.0" "librosa" "jiwer" "pyyaml"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.7 MB/s eta 0:00:00


In [2]:
# 2. Directorios y entorno de trabajo
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"
REPO_DIR = Path("/content/Proyecto-IA-Traductor")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU no disponible. Ejecutando en CPU.")
    print("TIP Colab: Para mayor velocidad, activa GPU en Colab (Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU).")

CUDA disponible: True
GPU: Tesla T4


## 2. Preparación de Muestras de Audio (FLEURS dev)

In [3]:
from io import BytesIO
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio

def decode_audio(audio: dict) -> tuple[np.ndarray, int]:
    """Decodifica un objeto Audio de datasets sin requerir torchcodec."""
    if audio.get("array") is not None and audio.get("sampling_rate") is not None:
        array = np.asarray(audio["array"], dtype=np.float32)
        sample_rate = int(audio["sampling_rate"])
    else:
        if audio.get("bytes") is not None:
            source = BytesIO(audio["bytes"])
        elif audio.get("path"):
            source = audio["path"]
        else:
            raise ValueError("El objeto de audio no contiene bytes, path ni array.")
        array, sample_rate = sf.read(source, dtype="float32", always_2d=False)
        array = np.asarray(array, dtype=np.float32)

    if array.ndim == 2:
        array = array.mean(axis=1, dtype=np.float32)
    return array, sample_rate

def get_transcription(sample: dict) -> str:
    return str(sample.get("raw_transcription") or sample.get("transcription") or "")

audio_dir = PROJECT_ROOT / "outputs/smoke_tests/audio"
audio_dir.mkdir(parents=True, exist_ok=True)

print("Cargando 3 muestras de FLEURS es_419 en streaming...")
stream = load_dataset("google/fleurs", "es_419", split="train", streaming=True).cast_column("audio", Audio(decode=False))
samples_stream = stream.shuffle(seed=42, buffer_size=100)

test_samples = []
for i, item in enumerate(samples_stream.take(3)):
    sample_id = f"fleurs_es_419_smoke_{i:02d}"
    audio_arr, sr = decode_audio(item["audio"])
    wav_path = audio_dir / f"{sample_id}.wav"
    if not wav_path.exists():
        sf.write(str(wav_path), audio_arr, sr, subtype="FLOAT")
    gt = get_transcription(item)
    test_samples.append({
        "sample_id": sample_id,
        "wav_path": str(wav_path),
        "ground_truth": gt,
        "duration_sec": round(len(audio_arr) / sr, 2),
    })

print(f"Muestras preparadas: {len(test_samples)}")

Cargando 3 muestras de FLEURS es_419 en streaming...


README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

Muestras preparadas: 3


## 3. Inferencia ASR con `openai/whisper-small`

In [4]:
import time
import jiwer
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Intentar importar desde módulo local si existe; de lo contrario usar implementación autónoma
try:
    from src.asr.transcribe import WhisperTranscriber
except (ImportError, ModuleNotFoundError):
    class WhisperTranscriber:
        """Transcriptor de voz a texto en español utilizando openai/whisper-small (autónomo en Colab)."""
        def __init__(
            self,
            model_name: str = "openai/whisper-small",
            device: str | torch.device | None = None,
            language: str = "es",
        ) -> None:
            self.model_name = model_name
            self.language = language
            self.task = "transcribe"
            if device is None:
                self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            else:
                self.device = torch.device(device)

            print(f"Cargando modelo ASR ({self.model_name}) en {self.device}...")
            self.processor = WhisperProcessor.from_pretrained(self.model_name)
            self.model = WhisperForConditionalGeneration.from_pretrained(self.model_name)
            self.model.to(self.device)
            self.model.eval()

            self.forced_decoder_ids = self.processor.get_decoder_prompt_ids(
                language=self.language,
                task=self.task,
            )

        def transcribe(self, audio_path: str) -> str:
            audio_array, sr = sf.read(str(audio_path), dtype="float32", always_2d=False)
            if audio_array.ndim == 2:
                audio_array = audio_array.mean(axis=1)
            if sr != 16_000:
                import librosa
                audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16_000)

            inputs = self.processor(audio_array, sampling_rate=16_000, return_tensors="pt")
            input_features = inputs.input_features.to(self.device)

            with torch.no_grad():
                predicted_ids = self.model.generate(
                    input_features,
                    forced_decoder_ids=self.forced_decoder_ids,
                )

            transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)
            return transcription[0].strip() if transcription else ""

transcriber = WhisperTranscriber()
print(f"Transcriptor listo en {transcriber.device}")

results = []
for s in test_samples:
    t0 = time.time()
    pred = transcriber.transcribe(s["wav_path"])
    dur = time.time() - t0
    wer = round(jiwer.wer(s["ground_truth"], pred) * 100, 2)
    cer = round(jiwer.cer(s["ground_truth"], pred) * 100, 2)
    results.append({
        "sample_id": s["sample_id"],
        "audio_file": s["wav_path"],
        "duration_sec": s["duration_sec"],
        "inference_time_sec": round(dur, 3),
        "ground_truth": s["ground_truth"],
        "prediction": pred,
        "wer": wer,
        "cer": cer,
    })

print("Inferencia completada.")

Cargando modelo ASR (openai/whisper-small) en cuda...


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


Transcriptor listo en cuda


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Inferencia completada.


## 4. Tabla de Resultados y Métricas (WER / CER)

In [5]:
import json
import pandas as pd

df_res = pd.DataFrame(results)[["sample_id", "duration_sec", "inference_time_sec", "ground_truth", "prediction", "wer", "cer"]]
pd.set_option("display.max_colwidth", None)
display(df_res)

mean_wer = round(sum(r["wer"] for r in results) / len(results), 2)
mean_cer = round(sum(r["cer"] for r in results) / len(results), 2)
mean_time = round(sum(r["inference_time_sec"] for r in results) / len(results), 3)

print("=" * 60)
print("SMOKE TEST — 3 MUESTRAS. NO ES UNA EVALUACIÓN GLOBAL DE WHISPER.")
print(f"WER promedio: {mean_wer}%")
print(f"CER promedio: {mean_cer}%")
print(f"Tiempo promedio de inferencia: {mean_time} s")
print("=" * 60)

# Guardar en outputs/smoke_tests/
output_json = {
    "model_name": "openai/whisper-small",
    "language": "es",
    "task": "transcribe",
    "device": str(transcriber.device),
    "num_samples": len(results),
    "disclaimer": "SMOKE TEST — 3 MUESTRAS. NO ES UNA EVALUACIÓN GLOBAL DE WHISPER.",
    "summary_metrics": {
        "mean_wer": mean_wer,
        "mean_cer": mean_cer,
        "mean_inference_time_sec": mean_time,
    },
    "samples": results,
}

out_file = PROJECT_ROOT / "outputs/smoke_tests/asr_results.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(output_json, f, indent=2, ensure_ascii=False)
print(f"Resultados guardados en: {out_file}")

,sample_id,duration_sec,inference_time_sec,ground_truth,prediction,wer,cer
0,fleurs_es_419_smoke_00,8.88,3.597,"Exactamente a las 8:46 a. m., la ciudad se volvió silencio, marcando el preciso instante en el que el primer avión alcanzó su objetivo.","Exactamente a las 8.46 a.m. la ciudad se volvió silencio, marcando de preciso instante en el que el primer avión alcanzó su objetivo.",16.67,3.70
1,fleurs_es_419_smoke_01,8.94,0.566,"Por lo general, las hembras se relacionan estrechamente entre sí, lo que las hace una gran familia de hermanas e hijas.","Por lo general, las hembras se relacionan estrechamente entre sí, lo que las hace una gran familia de hermanas e hijas.",0.00,0.00
2,fleurs_es_419_smoke_02,8.64,0.668,"Recuerde incluir los gastos adicionales, los impuestos de partida, el transporte terrestre, etc. para todos los lugares que visite fuera de África.","Recuerda incluir los gastos adicionales, los impuestos de partida, el transporte terrestre, etcétera para todos los lugares que visite fuera de África.",9.09,4.08


SMOKE TEST — 3 MUESTRAS. NO ES UNA EVALUACIÓN GLOBAL DE WHISPER.
WER promedio: 8.59%
CER promedio: 2.59%
Tiempo promedio de inferencia: 1.61 s
Resultados guardados en: /content/Proyecto-IA-Traductor/outputs/smoke_tests/asr_results.json
